1. Imports

In [1]:
import torch
import torch.nn as nn 
from transformers import AutoTokenizer, AutoModel

device = torch.device(
    "mps" if torch.backends.mps.is_available() else "cpu"
)

print("PyTorch", torch.__version__)
print("Device:", device)

PyTorch 2.14.0
Device: mps


2. Load Tokenizer

In [2]:
MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 32

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)
print("Tokenizer loaded:", MODEL_NAME)

Tokenizer loaded: distilbert-base-uncased


3. Load DistilBERT

In [4]:
text_encoder = AutoModel.from_pretrained(
    MODEL_NAME
)
text_encoder = text_encoder.to(device)
print(text_encoder)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


DistilBertModel(
  (embeddings): Embeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (transformer): Transformer(
    (layer): ModuleList(
      (0-5): 6 x TransformerBlock(
        (attention): DistilBertSelfAttention(
          (q_lin): Linear(in_features=768, out_features=768, bias=True)
          (k_lin): Linear(in_features=768, out_features=768, bias=True)
          (v_lin): Linear(in_features=768, out_features=768, bias=True)
          (out_lin): Linear(in_features=768, out_features=768, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
        (ffn): FFN(
          (dropout): Dropout(p=0.1, inplace=False)
          (lin1): Linear(in_features=768, out_features=3072, bias=Tru

In [5]:
print("Hidden size:", text_encoder.config.hidden_size)
print("Number of layers:", text_encoder.config.n_layers)
print("Number of attention heads:", text_encoder.config.n_heads)

Hidden size: 768
Number of layers: 6
Number of attention heads: 12


4. Test a Product Name

In [7]:
sample_text = "Wildcraft Unisex Orange Alpinist 55 Backpack"

encoded = tokenizer(
    sample_text,
    padding = "max_length",
    truncation= True,
    max_length = MAX_LENGTH,
    return_tensors = "pt"
)
print("Input IDs shape:", encoded["input_ids"].shape)
print("Attention mask shape:", encoded["attention_mask"].shape)

Input IDs shape: torch.Size([1, 32])
Attention mask shape: torch.Size([1, 32])


5. Move Inputs to MPS

In [8]:
input_ids = encoded["input_ids"].to(device)
attention_mask = encoded["attention_mask"].to(device)

print("Input IDs device:", input_ids.device)
print("Attension mask device:", attention_mask.device)

Input IDs device: mps:0
Attension mask device: mps:0


6. Forward Pass

In [11]:
with torch.no_grad():
    outputs = text_encoder(
        input_ids = input_ids,
        attention_mask= attention_mask
    )
print("Last hidden state shape:", outputs.last_hidden_state.shape)

Last hidden state shape: torch.Size([1, 32, 768])


7. Implement Masked Mean Pooling

In [12]:
def masked_mean_pooling(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(
        last_hidden_state.size()
    ).float()

    summed_embeddings = torch.sum(
        last_hidden_state * mask,
        dim =1
    )
    token_count = torch.clamp(
        mask.sum(dim=1),
        min = 1e-9
    )
    mean_embeddings = summed_embeddings / token_count
    return mean_embeddings

8. Generate a 768-Dimensional Embedding

In [13]:
with torch.no_grad():
    outputs = text_encoder(
        input_ids = input_ids,
        attention_mask = attention_mask
    )

text_embedding = masked_mean_pooling(
    outputs.last_hidden_state,
    attention_mask
)
print("Last hidden state shape:", outputs.last_hidden_state.shape)
print("Text embedding shape:", text_embedding.shape)

Last hidden state shape: torch.Size([1, 32, 768])
Text embedding shape: torch.Size([1, 768])


In [14]:
print("Embedding dtype:", text_embedding.dtype)
print("Embedding device:", text_embedding.device)
print("First 10 values:")
print(text_embedding[0][:10])

Embedding dtype: torch.float32
Embedding device: mps:0
First 10 values:
tensor([-0.1580, -0.2719,  0.2193,  0.3440,  0.3513, -0.0395, -0.0457, -0.0165,
        -0.0089, -0.1717], device='mps:0')


9. Verify Padding is Being Ignored

In [15]:
test_embeddings = torch.tensor(
    [
        [
            [1.0, 2.0],
            [3.0, 4.0],
            [100.0, 200.0]
        ]
    ],
    device = device
)

test_mask = torch.tensor(
    [
        [1, 1, 0]
    ],
    device = device
)
pooled = masked_mean_pooling(
    test_embeddings,
    test_mask
)
print("Pooled result:", pooled)

Pooled result: tensor([[2., 3.]], device='mps:0')


10. Create a Reusable Text Encoder

In [54]:
class DistilBERTTextEncoder(nn.Module):

    def __init__(
        self,
        model_name="distilbert-base-uncased"
    ):
        super().__init__()

        self.transformer = AutoModel.from_pretrained(
            model_name,
            attn_implementation="eager"
        )

        self.hidden_size = (
            self.transformer.config.hidden_size
        )

    def masked_mean_pooling(
        self,
        last_hidden_state,
        attention_mask
    ):
        mask = (
            attention_mask
            .unsqueeze(-1)
            .expand(last_hidden_state.size())
            .float()
        )

        summed_embeddings = torch.sum(
            last_hidden_state * mask,
            dim=1
        )

        token_count = torch.clamp(
            mask.sum(dim=1),
            min=1e-9
        )

        mean_embeddings = (
            summed_embeddings / token_count
        )

        return mean_embeddings

    def forward(
        self,
        input_ids,
        attention_mask
    ):

        outputs = self.transformer(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        text_embedding = self.masked_mean_pooling(
            outputs.last_hidden_state,
            attention_mask
        )

        return text_embedding

In [20]:
text_encoder_model = DistilBERTTextEncoder(
    MODEL_NAME
).to(device)

text_encoder_model.eval()

print("Text embedding dimension:",
      text_encoder_model.hidden_size)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Text embedding dimension: 768


11. Test the Complete Encoder

In [21]:
with torch.no_grad():
    text_embedding = text_encoder_model(
        input_ids,
        attention_mask
    )

print("Input IDs:", input_ids.shape)
print("Attention mask:", attention_mask.shape)
print("Text embedding:", text_embedding.shape)

Input IDs: torch.Size([1, 32])
Attention mask: torch.Size([1, 32])
Text embedding: torch.Size([1, 768])


### Build Text Embedding

- Loaded pretrained DistilBERT
- Implemented attention-mask-aware mean pooling
- Converted token-level output [1, 32, 768] to [1, 768]
- Verified padding tokens are ignored
- Created reusable DistilBERTTextEncoder
- Verified output dimension: 768
- Verified dtype: float32
- Verified MPS execution

12. Create the Text Classifier

In [55]:
class TextClassifier(nn.Module):

    def __init__(
        self,
        model_name="distilbert-base-uncased",
        num_classes=20
    ):
        super().__init__()

        self.text_encoder = DistilBERTTextEncoder(
            model_name
        )

        self.classifier = nn.Linear(
            self.text_encoder.hidden_size,
            num_classes
        )

    def forward(
        self,
        input_ids,
        attention_mask
    ):

        text_embedding = self.text_encoder(
            input_ids,
            attention_mask
        )

        logits = self.classifier(
            text_embedding
        )

        return logits

13. Create the Model

In [56]:
MODEL_NAME = "distilbert-base-uncased"
NUM_CLASSES = 20

text_classifier = TextClassifier(
    model_name=MODEL_NAME,
    num_classes=NUM_CLASSES
).to(device)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


14. Freeze DistilBERT Initially

In [57]:
for param in text_classifier.text_encoder.transformer.parameters():
    param.requires_grad = False

for param in text_classifier.classifier.parameters():
    param.requires_grad = True

15. Verify Trainable Parameters

In [67]:
total_params = sum(
    p.numel()
    for p in text_classifier.parameters()
)

trainable_params = sum(
    p.numel()
    for p in text_classifier.parameters()
    if p.requires_grad
)

print("Total parameters:", total_params)
print("Trainable parameters:", trainable_params)

Total parameters: 66378260
Trainable parameters: 15380


16. Verify Forward Pass

In [59]:
text_classifier.eval()

with torch.no_grad():
    logits = text_classifier(
        input_ids,
        attention_mask
    )

print("Input IDs shape:", input_ids.shape)
print("Attention mask shape:", attention_mask.shape)
print("Logits shape:", logits.shape)

Input IDs shape: torch.Size([1, 32])
Attention mask shape: torch.Size([1, 32])
Logits shape: torch.Size([1, 20])


### Build Text-Only Classification Model

- Created TextClassifier using DistilBERT
- DistilBERT hidden representation: 768 dimensions
- Added 768 → 20 classification head
- Frozen pretrained DistilBERT backbone
- Classification head remains trainable
- Total parameters: 66,378,260
- Trainable parameters: 15,380
- Verified forward pass
- Verified output shape: [1, 20]


17. Loss Function

In [60]:
criterion = nn.CrossEntropyLoss()

print("Loss function:", criterion)

Loss function: CrossEntropyLoss()


18. Optimizer

In [61]:
optimizer = torch.optim.Adam(
    text_classifier.classifier.parameters(),
    lr=1e-3
)
print("Optimizer:", optimizer)

Optimizer: Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0
)


19.Scheduler

In [63]:
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.1,
    patience=2
)

print("Scheduler:", scheduler)

Scheduler: <torch.optim.lr_scheduler.ReduceLROnPlateau object at 0x12899e5d0>


In [64]:
print("Trainable parameters:")

for name, param in text_classifier.named_parameters():
    if param.requires_grad:
        print(name, param.shape)

Trainable parameters:
classifier.weight torch.Size([20, 768])
classifier.bias torch.Size([20])


In [65]:
optimizer_params = sum(
    p.numel()
    for group in optimizer.param_groups
    for p in group["params"]
)

print("Parameters passed to optimizer:", optimizer_params)

Parameters passed to optimizer: 15380


### Configure Text Model Training

- Loss function: CrossEntropyLoss
- Optimizer: Adam
- Learning rate: 0.001
- Scheduler: ReduceLROnPlateau
- DistilBERT backbone: Frozen
- Classification head: Trainable
- Trainable parameters: 15,380
- Verified optimizer parameter count

20. Training Function

In [68]:
def train_text_one_epoch(
    model,
    dataloader,
    criterion,
    optimizer,
    device
):

    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for batch in dataloader:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()

        # Forward pass
        logits = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        # Calculate loss
        loss = criterion(
            logits,
            labels
        )

        # Backpropagation
        loss.backward()

        # Update classifier
        optimizer.step()

        # Statistics
        batch_size = labels.size(0)

        running_loss += (
            loss.item() * batch_size
        )

        predictions = logits.argmax(
            dim=1
        )

        correct += (
            predictions == labels
        ).sum().item()

        total += batch_size

    epoch_loss = running_loss / total
    epoch_accuracy = correct / total

    return epoch_loss, epoch_accuracy

21. Validation Function

In [70]:
def validate_text_one_epoch(
    model,
    dataloader,
    criterion,
    device
):

    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():

        for batch in dataloader:

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)

            # Forward pass
            logits = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            # Loss
            loss = criterion(
                logits,
                labels
            )

            batch_size = labels.size(0)

            running_loss += (
                loss.item() * batch_size
            )

            predictions = logits.argmax(
                dim=1
            )

            correct += (
                predictions == labels
            ).sum().item()

            total += batch_size

    epoch_loss = running_loss / total
    epoch_accuracy = correct / total

    return epoch_loss, epoch_accuracy

22. Training History

In [71]:
NUM_EPOCHS = 5

text_history = {
    "train_loss": [],
    "train_accuracy": [],
    "val_loss": [],
    "val_accuracy": [],
    "learning_rate": []
}

23. Train

In [49]:
%run ./02_Data_Preparation.ipynb

Device: mps
Train: (29901, 6)
Validation: (6435, 6)
Test: (6459, 6)
../data/raw/fashion-product-images-small/images/15970.jpg
Exists: True
Image size: (60, 80)
Image mode: RGB
Image mode: RGB
Tensor shape: torch.Size([3, 224, 224])
Tensor dtype: torch.float32
Tenosr min: -2.0836544036865234
Tensor max: 2.640000104904175
Dataset size: 29901
Image shape: torch.Size([3, 224, 224])
Image dtype: torch.float32
Label: tensor(17)
Label dtype: torch.int64
index 0: shape=torch.Size([3, 224, 224]), label=17
index 100: shape=torch.Size([3, 224, 224]), label=17
index 1000: shape=torch.Size([3, 224, 224]), label=13
index 5000: shape=torch.Size([3, 224, 224]), label=17
index 10000: shape=torch.Size([3, 224, 224]), label=0
Train datasets: 29901
Validation datasets: 6435
Test dataset: 6459
Images shape: torch.Size([32, 3, 224, 224])
Labels shape: torch.Size([32])
Images dtype: torch.float32
Label dtype: torch.int64
Labels: tensor([ 3,  9,  0, 17, 15, 13, 17, 17, 17, 15,  0, 19, 17, 13, 18, 17, 17,  0,


In [73]:
best_val_loss = float("inf")

for epoch in range(NUM_EPOCHS):

    train_loss, train_accuracy = train_text_one_epoch(
        text_classifier,
        multimodal_train_loader,
        criterion,
        optimizer,
        device
    )

    val_loss, val_accuracy = validate_text_one_epoch(
        text_classifier,
        multimodal_val_loader,
        criterion,
        device
    )

    # Scheduler
    scheduler.step(val_loss)

    current_lr = optimizer.param_groups[0]["lr"]

    # Store history
    text_history["train_loss"].append(
        train_loss
    )

    text_history["train_accuracy"].append(
        train_accuracy
    )

    text_history["val_loss"].append(
        val_loss
    )

    text_history["val_accuracy"].append(
        val_accuracy
    )

    text_history["learning_rate"].append(
        current_lr
    )

    print(
        f"Epoch [{epoch + 1}/{NUM_EPOCHS}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_accuracy:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_accuracy:.4f} | "
        f"LR: {current_lr:.6f}"
    )

    # Save best model
    if val_loss < best_val_loss:

        best_val_loss = val_loss

        torch.save(
            text_classifier.state_dict(),
            "../models/distilbert_text_classifier_best.pth"
        )

        print("  ✓ Best text model saved")

Epoch [1/5] | Train Loss: 0.6172 | Train Acc: 0.8527 | Val Loss: 0.3389 | Val Acc: 0.9159 | LR: 0.001000
  ✓ Best text model saved
Epoch [2/5] | Train Loss: 0.2970 | Train Acc: 0.9340 | Val Loss: 0.2214 | Val Acc: 0.9543 | LR: 0.001000
  ✓ Best text model saved
Epoch [3/5] | Train Loss: 0.2142 | Train Acc: 0.9516 | Val Loss: 0.1702 | Val Acc: 0.9610 | LR: 0.001000
  ✓ Best text model saved
Epoch [4/5] | Train Loss: 0.1719 | Train Acc: 0.9605 | Val Loss: 0.1416 | Val Acc: 0.9681 | LR: 0.001000
  ✓ Best text model saved
Epoch [5/5] | Train Loss: 0.1463 | Train Acc: 0.9663 | Val Loss: 0.1229 | Val Acc: 0.9728 | LR: 0.001000
  ✓ Best text model saved


### Train Text-Only Baseline

- Used multimodal_train_loader for training
- Used multimodal_val_loader for validation
- Image tensors were completely ignored
- DistilBERT backbone remained frozen
- Classification head was trained
- Epochs: 5
- Best validation accuracy: 97.28%
- Best validation loss: 0.1229
- Best model checkpoint saved